In [1]:
import torch
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
from datasets import load_dataset
DATASET_NAME="mlabonne/guanaco-llama2-1k"
dataset=load_dataset(DATASET_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1000
    })
})


In [4]:
print(dataset['train'])
training_dataset=dataset['train']

Dataset({
    features: ['text'],
    num_rows: 1000
})


In [5]:
print("SAMPLE DATA IN DATASET:")
for i in range(5):
  print(training_dataset[i])

SAMPLE DATA IN DATASET:
{'text': '<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'}
{'text': '<s>[INST] Самый великий человек из всех живших на планете? [/INST] Для начала нужно выбрать к

In [6]:
training_dataset[11]

{'text': '<s>[INST] write me a 1000 words essay about deez nuts. [/INST] The Deez Nuts meme first gained popularity in 2015 on the social media platform Vine. The video featured a young man named Rodney Bullard, who recorded himself asking people if they had heard of a particular rapper. When they responded that they had not, he would respond with the phrase "Deez Nuts" and film their reactions. The video quickly went viral, and the phrase became a popular meme. \n\nSince then, Deez Nuts has been used in a variety of contexts to interrupt conversations, derail discussions, or simply add humor to a situation. It has been used in internet memes, in popular music, and even in politics. In the 2016 US presidential election, a 15-year-old boy named Brady Olson registered as an independent candidate under the name Deez Nuts. He gained some traction in the polls and even made appearances on national news programs.\n\nThe Deez Nuts meme has had a significant impact on popular culture. It has b

In [7]:
MODEL_NAME="distilgpt2"
import transformers
from transformers import AutoModelForCausalLM #Called causal because it generates one token at a time
from transformers import AutoTokenizer

model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,device_map="auto")#Downloads weights and accelerate puts them in the GPU in the right place using device map
model.config.use_cache=True #KV Caching(Caching can mess up gradients while training)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True)#Downloads vocabulary rules(subword tokenization rules for gpt2)

# --- The Padding Fix ---
#If one tensor has 5 words and the other has 10, gpt2 crashes, hence we need a fixed size padding
tokenizer.pad_token = tokenizer.eos_token#use end of sequence as a dummy padding
tokenizer.padding_side = 'right'#add padded tokens to the end of the sentence
tokenizer.pad_token_id = tokenizer.eos_token_id#so that id numbers match

# --- The Generation Settings ---
generation_configuration = model.generation_config
generation_configuration.pad_token_id = tokenizer.eos_token_id
generation_configuration.eos_token_id = tokenizer.eos_token_id
model.generation_config.do_sample = True
generation_configuration.max_new_tokens = 1024
generation_configuration.temperature = 0.7 #Controls the "chaos" or creativity. At 0.0, it always picks the most obvious word (robotic). At 1.0, it takes risks (creative, but sometimes hallucinates).
generation_configuration.top_p = 0.9 #Instead of a fixed number of words, it picks words until their combined probabilities hit p
generation_configuration.top_k = 20 #It forces the AI to only look at the top K most likely next tokens and completely ignores the rest.

#First, top_k acts as a safety net: It chops off everything below the top 20 words, guaranteeing the AI will never accidentally pick "xylophone" just because of a weird math glitch.
#Then, top_p acts as the dynamic filter: It looks at those 20 remaining words and dynamically slices them down further until it captures 90% of the probability mass.

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [8]:
def generate(prompt):
  encoded=tokenizer.encode(prompt,add_special_tokens=True,return_tensors="pt").to(device)#encoding the prompt into tokens and adds bos and eos tokens
  out=model.generate(input_ids=encoded,repetition_penalty=2.0,do_sample=True)#penalty for the model to not generate same token twice
  string_decoded = tokenizer.decode(out[0].tolist(), clean_up_tokenization_spaces=True)
  print(string_decoded)
#The model spits out "out", which is just a giant list of numbers containing both the prompt and its new generated words
#out[0].tolist(): Grabs the first sequence of generated numbers and turns it back into a standard Python list.
#tokenizer.decode: Translates those numbers back into English words.
#clean_up_tokenization_spaces=True: When the tokenizer chops up words and punctuation, it sometimes leaves awkward spaces (like "Hello , world ."). This cleans it up so it looks like natural human typing ("Hello, world.").

In [9]:
generate('hi my name is')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


hi my name is the only person who can understand.”<|endoftext|>


In [10]:
from trl import SFTTrainer,SFTConfig
from transformers import TrainingArguments
training_args = SFTConfig(
    # --- THE MEMORY HACK ---
    per_device_train_batch_size=2,      # process only 2 examples(one mini batch) at a time (Saves GPU RAM)
    gradient_accumulation_steps=8,      # wait for 8 mini-batches to finish before updating the brain
    # effective Batch Size = 2 x 8 = 16 # if we directly keep batch_size=16, then GPU crashes
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    output_dir="logs",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_length=512
)

trainer = SFTTrainer(
model=model,
train_dataset=training_dataset,
processing_class=tokenizer ,
args=training_args,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.432963
20,3.438644
30,3.228788
40,3.271673
50,3.192391
60,3.254916


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=3.2960495267595564, metrics={'train_runtime': 68.9489, 'train_samples_per_second': 14.503, 'train_steps_per_second': 0.914, 'total_flos': 114956280594432.0, 'train_loss': 3.2960495267595564})

In [13]:
generate("how are you")

how are you trying to or and.. ) " () false true exact signature Signature
-
<|endoftext|>
